In [75]:
import pandas as pd
import numpy as np

seeds = [2]
# terms = ['HALLMARK_INTERFERON_ALPHA_RESPONSE','HALLMARK_INTERFERON_GAMMA_RESPONSE',
#         'HALLMARK_IL6_JAK_STAT3_SIGNALING','HALLMARK_INFLAMMATORY_RESPONSE',
#         'HALLMARK_TGF_BETA_SIGNALING','HALLMARK_DNA_REPAIR','HALLMARK_ALLOGRAFT_REJECTION',
#         'HALLMARK_COMPLEMENT', 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION',
#         'HALLMARK_WNT_BETA_CATENIN_SIGNALING']
for seed in seeds:
    GSEA_results = {}
    synthetic_datasets = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",
                              f"synthpop_{seed}",f"tvae_{seed}"]
    or_gsea = pd.read_csv(f'NivoBenefit/Seed_{seed}/GSEA_Origin.csv')
    # or_gsea = or_gsea[or_gsea['Term'].isin(terms)]
    GSEA_results['Origin'] = or_gsea[['Term','NES']]
    for syntheticdata in synthetic_datasets:
        try:
            
            syn_gsea =  pd.read_csv(f'NivoBenefit/Seed_{seed}/GSEA_{syntheticdata}.csv')
            # syn_gsea = syn_gsea[syn_gsea['Term'].isin(terms)]
            GSEA_results[syntheticdata] = syn_gsea[['Term','NES']]
        except:
            continue

In [76]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def proportion_direction_agreement_with_permutation(
    data_dict,
    origin_key="Origin",
    term_col="Term",
    score_col="NES score",
    drop_zero=True,
    n_perm=1000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    # === Chuẩn hoá origin ===
    origin = data_dict[origin_key][[term_col, score_col]].dropna().copy()
    origin[term_col] = origin[term_col].astype(str)
    origin = origin.loc[origin[score_col].abs().sort_values(ascending=False).index]
    origin = origin.drop_duplicates(subset=term_col, keep="first")
    origin["sign_origin"] = np.sign(origin[score_col]).astype(int)

    results = []

    # === Loop synthetic ===
    for k, df in data_dict.items():
        if k == origin_key:
            continue

        syn = df[[term_col, score_col]].dropna().copy()
        syn[term_col] = syn[term_col].astype(str)
        syn = syn.loc[syn[score_col].abs().sort_values(ascending=False).index]
        syn = syn.drop_duplicates(subset=term_col, keep="first")
        syn["sign_syn"] = np.sign(syn[score_col]).astype(int)

        merged = origin[[term_col, "sign_origin"]].merge(
            syn[[term_col, "sign_syn"]], on=term_col, how="inner"
        )

        if drop_zero:
            merged = merged[(merged["sign_origin"] != 0) & (merged["sign_syn"] != 0)]

        n_overlap = len(merged)
        if n_overlap == 0:
            results.append({
                "synthetic_key": k,
                "overlap_pathways_used": 0,
                "agree_direction_count": 0,
                "proportion_direction_agreement": np.nan,
                "p_permute": np.nan,
                "composite_score": np.nan
            })
            continue

        # observed proportion
        observed_prop = (merged["sign_origin"] == merged["sign_syn"]).mean()

        # === permutation test: shuffle synthetic signs ===
        greater_equal = 0
        syn_signs = merged["sign_syn"].values
        origin_signs = merged["sign_origin"].values

        for _ in range(n_perm):
            shuffled = rng.permutation(syn_signs)
            prop_perm = (origin_signs == shuffled).mean()
            if prop_perm >= observed_prop:
                greater_equal += 1

        # p-value (with pseudo count)
        p_perm = (greater_equal + 1) / (n_perm + 1)

        composite = observed_prop * (-np.log10(p_perm))

        results.append({
            "synthetic_key": k,
            "overlap_pathways_used": n_overlap,
            "agree_direction_count": int(observed_prop * n_overlap),
            "proportion_direction_agreement": observed_prop,
            "p_permute": p_perm,
            "composite_score": composite,
        })

    return pd.DataFrame(results)


In [77]:
df_agreement = proportion_direction_agreement_with_permutation(
    GSEA_results,
    origin_key="Origin",
    term_col="Term",
    score_col="NES",
)
df_agreement.to_csv(f'NivoBenefit/Seed_{seed}/DirectionAgreement.csv')
df_agreement

,synthetic_key,overlap_pathways_used,agree_direction_count,proportion_direction_agreement,p_permute,composite_score
0,avatarsk5_2,50,30,0.60,1.000000,-0.000000
1,avatarsk10_2,50,35,0.70,0.047952,0.923435
2,ctgan_2,50,26,0.52,0.734266,0.069756
3,gaussiancopula_2,50,30,0.60,0.863137,0.038352
4,synthpop_2,50,23,0.46,0.288711,0.248187
